In [2]:
import numpy as np
import yfinance as yf
import pandas as pd
import os
import matplotlib.pyplot as plt
import torch
import cvxpy as cp
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim
from pykalman import KalmanFilter

In [24]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()
        try:
            tmp_data = yf.Ticker(clean_ticker) \
                          .history(
                              start=start,
                              end=end,
                              interval=interval
                          )

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue

            tmp_data.index = pd.to_datetime(tmp_data.index)\
                                    .tz_localize(None)\
                                    .normalize()

            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
            data[clean_ticker] = tmp_data
        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError(
            "No data was successfully downloaded  \
              for any of the provided tickers."
        )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
      self,
      indices:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    macros=["^TNX", "^FVX", "DX-Y.NYB", "^TRCCRB"]
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"
    macros_path = f"yields_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = self.download(indices, start, end, interval)
      bench_data = self.download([benchmark], start, end, interval)["Close"]
      macro_data = self.download(macros, start, end, interval)["Close"]

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)
      macro_data.to_parquet(macros_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data = pd.read_parquet(benchmark_path)
      macro_data = pd.read_parquet(macros_path)

    returns_raw = df.pct_change().dropna()
    benchmark = bench_data.pct_change().dropna()
    self.universe = returns_raw.columns

    return returns_raw, benchmark, macro_data

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [28]:
class FeatureGenerator:
  def __init__(self, feature_params, kalman_params, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug
    self.kalman_params = kalman_params
    self.feature_params = feature_params

  def generate_gk_er(self, data):
    hl = np.square(np.log(data["High"]/data["Low"]))
    co = np.square(np.log(data["Close"]/data["Open"]))
    GK = (0.5*hl - (2*np.log(2))*co)

    return GK

  def generate_kalman_outputs(self, log_prices):
    kf_dict = {}
    dt = self.kalman_params["dt"]
    trans_mat = np.array(
        [
            [1, 1, dt],
            [0, 1, 1],
            [0, 0, 1]
        ]
    )
    obs_mat = np.array(
        [
            [1, 0, 0]
        ]
    )

    cwna_mtx = np.array(
        [
            [dt**5/20, dt**4/8, dt**3/6],
            [dt**4/8, dt**3/4, dt**2/2],
            [dt**3/6, dt**2/2, dt]
        ]
    )

    obs_cov = [[1.0]]
    init_cov = np.zeros_like(cwna_mtx)
    np.fill_diagonal(init_cov, [1]+[10]*(len(np.diag(cwna_mtx))-1))

    for col in log_prices.columns:
      init_state = np.array([log_prices[col][0], 0, 0])

      for i in ["Fast", "Mid", "Slow"]:

        trans_cov = self.kalman_params[i]*cwna_mtx
        col_name = f"{col}{i}"
        kf = KalmanFilter(
          transition_matrices=trans_mat,
          observation_matrices=obs_mat,
          transition_covariance=trans_cov,
          observation_covariance=obs_cov,
          initial_state_mean=init_state,
          initial_state_covariance=init_cov
        )

        state_means, _ = kf.filter(log_prices)

        denoised_values = state_means[:, 0]
        momentum_values = state_means[:, 1]
        acceleration_values = state_means[:, 2]

        kf_dict[col_name + "Velocity"] = momentum_values.values
        kf_dict[col_name + "Acceleration"] = acceleration_values.values

    return kf_dict

  def get_betas(self, X, y):
    cov = np.cov(X, y, rowvar=False)[0, 1]
    var = np.var(X, ddof=1)
    return cov / var

  def get_macro_features(self, macro_data, market_returns, window=30):
    spread = macro_data["^TNX"] - macro_data["^FVX"]
    spread_change = spread.pct_change()
    usd_idx_change = macro_data["DX-Y.NYB"].pct_change()
    cmd_idx_change = macro_data["^TRCCRB"].pct_change()

    macro_df = pd.concat(
      [spread_change, usd_idx_change, cmd_idx_change],
      axis=1
    )
    macro_df.columns = ['spread_change', 'usd_idx_change', 'cmd_idx_change']

    combined = pd.concat([macro_df, market_returns.rename('market')], axis=1).dropna()

    rolling_betas = pd.DataFrame(index=combined.index)

    for col in macro_df.columns:
      betas = []
      for i in range(len(combined)):
        if i < window - 1:
          betas.append(np.nan)
        else:
          window_data = combined.iloc[i - window + 1 : i + 1]
          X = window_data['market'].values
          y = window_data[col].values
          betas.append(self.get_betas(X, y))

      rolling_betas[f'{col}_beta'] = betas

    return rolling_betas

  def generate_ex_R_features(self, data):
    ex_r_features = {}
    cov_features = {}

    gk_vol = self.garman_klass_vol(data)/np.log(data["Volume"]+1)
    for i in gk_vol.columns:
      ex_r_features["GKEffRatio"+i] = gk_vol[i].values


    kalman_outputs = self.generate_kalman_outputs(np.log(data["Close"]))

    ex_r_features |= kalman_outputs

    ex_R_target = np.log(1 + data.pct_change().dropna())

    ex_r_features = pd.DataFrame(ex_r_features, index=data.index)

    cmm = ex_R_target.index.intersection(ex_r_features.index)
    ex_R_target_aligned = ex_R_target.loc[cmm]
    ex_r_features_aligned = ex_r_features.loc[cmm]

    return ex_r_features_aligned, ex_R_target_aligned

  def generate_cov_features(self, data, yield_data, window=30):
    cov_features = {}

    gk_vol = self.garman_klass_vol(data)/np.log(data["Volume"]+1)
    for i in gk_vol.columns:
      cov_features["GKVol"+i] = gk_vol[i].values

    cov_features = pd.DataFrame(cov_features, index=data.index)
    macro_betas = self.get_macro_betas(data, yield_data, window)

    cov_features_full = pd.concat([cov_features, macro_betas])

    cov_target = data["Close"].pct_change().dropna().rolling(window).cov()

    return cov_features, cov_target


In [6]:
class InputGenerator(nn.Module):
  def init(self, debug):
    self.debug = debug
    # use NLL loss

  def calculate_pvr(self, pred_ex_r, hist_ex_r):
    pass

  def generate_ex_returns(self, data):
    pass

  def generate_fwd_cov(self, data):
    pass

  def forward(self, data):
    pass


In [7]:
class Optimizer:
  def __init__(
      self,
      turnover_penalty_weight=0.5,
      risk_aversion=3.0,
      min_weight_threshold=0.01,
      debug:bool=False, **kwargs
    ):

    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug
    self.turnover_penalty = turnover_penalty_weight
    self.risk_aversion = risk_aversion
    self.min_weight_threshold = min_weight_threshold

  def optimize_portfolio(
    self,
    ex_returns,
    fwd_cov,
    w_prev=None
  ):
    S, N = ex_returns.shape
    R = ex_returns.values

    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    ex_R = ex_returns.values @ w

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0,
    ]

    if w_prev is not None:
        w_prev_arr = np.asarray(w_prev, dtype=float)
        if w_prev_arr.shape[0] != N:
            raise ValueError(
                f"w_prev length {w_prev_arr.shape[0]} != current N={N}. "
                "Caller must align w_prev to the current filtered universe."
            )
        turnover_penalty = self.turnover_penalty_weight * cp.norm1(w - w_prev_arr)
    else:
        turnover_penalty = 0.0

    ptf_risk = cp.sqrt(cp.quad_form(w, fwd_cov))


    obj_fn = cp.Maximize(ex_R - self.risk_aversion * ptf_risk - turnover_penalty)
    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL)

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    return w

  def process_w(self, w):
    weights = np.array(w.value)
    weights[weights < self.min_weight_threshold] = 0.0

    total_w = np.sum(weights)
    if total_w > 0:
        weights_recalc = weights / total_w

    return weights_recalc

In [8]:
class Portfolio(Optimizer, DataStore, FeatureGenerator):
  def __init__(self, recalc_window, optim_params, returns_gen_params, debug:bool=False, **kwargs):
    super().__init__(
        recalc_window,
        **optim_params,
        **returns_gen_params,
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def convert_to_simple(self, log_returns, prev_returns):
    arithm_returns = np.exp(log_returns) - 1 + 0.5*(prev_returns.std())**2
    return arithm_returns

  def generate_ex_returns(self, ex_r_data, ex_r_params):
    # collect iputs
    # load model
    # predict E[r]
    # convert r to simple with convexity adjustment
    # get prediction volatility ratio
    pass

  def generate_fwd_cov(self, cov_data, cov_params):
    # collect iputs
    # load model
    # predict Σ
    pass

  def generate_features(self):
    ex_r_data = self.generate_ex_r_features()
    cov_data = self.generate_cov_features()

    return ex_r_data, cov_data

  def optimize(self, ex_returns, fwd_cov, w_prev=None):
    w = self.optimize_portfolio(ex_returns, fwd_cov, w_prev=w_prev)
    w_clean = self.process_w(w, self.min_weight_threshold)
    return w_clean

  def train(self, ex_r_data, cov_data):
    pass

In [ ]:
class Backtest(Portfolio, DataStore):
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug